# Finding the Higgs boson in real diphoton data

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/higgs_boson)

About 95 000 photon pairs from 10 fb$^{-1}$ of real 13 TeV ATLAS collisions (ATLAS Open Data, CC0) are binned in diphoton mass $m_{\gamma\gamma}$ and fitted with autograd: a smooth falling background plus a narrow Higgs peak. The profile likelihood ratio gives $m_H$, the signal strength $\mu$ and the local significance $Z = \sqrt{q_0}$, a batch of background-only toy experiments shows the look-elsewhere effect, and the Mexican-hat potential $V = -\mu^2|\phi|^2 + \lambda|\phi|^4$ turns the measured mass into the Higgs self-coupling $\lambda = m_H^2/2v^2$.

**Vector form**

$$\nu_n(\mu, m_H, \theta) = w_n\,e^{\theta_k x_n^k} + \mu\,N_{SM}\,S_n(m_H),\qquad q_0 = -2\ln\frac{L(0, \hat{\hat\theta})}{L(\hat\mu, \hat m_H, \hat\theta)},\qquad Z = \sqrt{q_0},\qquad m_H^2 = 2\lambda v^2$$

**Index (tensor) form**

$$B_{bn} = w_n\exp(X_{nk}\theta_{bk}),\qquad J_{bnj} = \frac{\partial\nu_{bn}}{\partial p_j},\qquad g_{bj} = J_{bnj}\Big(1 - \frac{k_{bn}}{\nu_{bn}}\Big),\qquad I_{bjl} = \frac{J_{bnj}J_{bnl}}{\nu_{bn}},\qquad p_{bj} \leftarrow p_{bj} - [I_b^{-1}]_{jl}\,g_{bl}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $k_n$ | `load_histogram(path, m_min, m_max)` | (N,) | real diphoton counts per 0.5 GeV bin; any CSV with m_low, m_high, events works |
| $X_{nk} = x_n^k$ | `design(centers, m_min, m_max, degree)` | (N, K) | powers of the scaled mass $x \in [0, 1]$; the background is $\exp(X\theta)$ |
| $S_n(m_H)$ | `signal_shape(edges, m_h, sigma)` | (M, N) | Gaussian peak integrated over each bin (erf), width $\sigma$ fitted to the ATLAS simulation |
| $-\ln L$ | `(nu - counts * log(nu)).sum(-1)` | (B,) | Poisson likelihood of each histogram, minimised by Adam + L-BFGS through autograd |
| $g_{bj},\ I_{bjl}$ | `einsum('bnj,bnl,bn->bjl', jac, jac, 1 / nu)` | (B, K+1, K+1) | repeated $n$ ⇒ summed: Fisher scoring fits every toy × every mass at once |
| $q_0$ | `q0_scan(counts, x, edges, widths, masses, sigma)` | (B, M) | profile likelihood ratio, 0 when the fitted signal is negative |
| $C = H^{-1}$ | `autograd.functional.hessian(nll, p)` | (K+2, K+2) | errors on $m_H$ and $\mu$ from the curvature of $-\ln L$ |
| $V(\phi)$ | `mexican_hat(phi, mu2, lam)` | (3, R) | potential along three rays $\phi = r\,e^{i\alpha}$ of the complex plane: the same curve for every phase, a ring of vacua at $/\phi/ = v/\sqrt2$ |

In July 2012 ATLAS and CMS announced a new boson near 125 GeV, seen above all in two channels: two photons and four leptons. The 2013 Nobel Prize in Physics went to Englert and Higgs for the mechanism it confirmed. Here the photon channel is redone on real data. The Higgs cannot decay to photons directly (they are massless), only through loops of W bosons and top quarks, so this is a rare decay ($\mathcal B \approx 0.23\%$) on top of a huge smooth background of ordinary photon pairs. With the 10 fb$^{-1}$ in this file, the result is $m_H \approx 124.2 \pm 1.0$ GeV (PDG 2024: $125.20 \pm 0.11$), $\mu \approx 1.0 \pm 0.4$ (the Standard Model rate) and a local significance of about $2.7\sigma$ (about $3.4\sigma$ expected). That is an excess, not a discovery: the 2012 discoveries used several event categories, both channels and both experiments to pass $5\sigma$. The scan is a useful warning too: in this small sample a background fluctuation near 143 GeV is almost as large ($2.6\sigma$ local) as the real Higgs at 124 GeV. This is the look-elsewhere effect. A bump somewhere in a wide window is likely, so the toys estimate how often background alone produces one: about 13% of background-only scans beat the 125 GeV excess somewhere, and the Gross–Vitells upcrossing formula gives a global p-value near 0.1. Background-only toys at fixed mass follow $\tfrac12\delta(0) + \tfrac12\chi^2_1$ (Wilks/Chernoff), which is why $Z = \sqrt{q_0}$. The Mexican hat: with $v = (\sqrt2 G_F)^{-1/2} = 246.2$ GeV from the muon lifetime, the measured mass fixes $\lambda = m_H^2/2v^2 \approx 0.13$; the radial wiggle about the vacuum ring is the Higgs boson and the flat direction along the ring becomes the longitudinal W and Z. Data: ATLAS Open Data 13 TeV, 2020 release, GamGam samples (CERN Open Data Portal record 15006, doi:10.7483/OPENDATA.ATLAS.B5BJ.3SGS, licence CC0; 2016 data periods A–D). Selection as in the ATLAS outreach H → γγ notebook: photon trigger, two tight isolated photons with $p_T > 40$ and 30 GeV, $|\eta| < 2.37$ outside 1.37–1.52. The sm_signal column is the matching ATLAS simulation (ggF, VBF, WH, ZH, ttH at 125 GeV) weighted to 10 fb$^{-1}$, which gives $N_{SM} \approx 406$ events and the peak width. Simplifications: one inclusive category, a Gaussian peak (the real one has tails, which is why the fitted width of about 2.3 GeV is larger than the core resolution), no systematic uncertainties, and an exponential-polynomial background (try the degree to see the spurious-signal sensitivity).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/higgs_boson/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lo, hi = params["m_min"], params["m_max"]
    data = tensors.load_histogram(tensors.DATA, lo, hi, dev)
    x = tensors.design(data["centers"], lo, hi, params["degree"])  # (N, K)
    n_sm = data["sm_signal"].sum().item()
    print(f"{int(data['counts'].sum().item())} diphoton events in {lo:g}–{hi:g} GeV (ATLAS Open Data, 10 fb⁻¹, 13 TeV)")

    # 1. the peak width from the ATLAS simulation, then the free-mass fit to the data
    m_sim, sigma = solve.fit_resolution(data["edges"], data["sm_signal"])
    print(f"simulated H → γγ: peak at {m_sim:.2f} GeV, Gaussian width σ = {sigma:.2f} GeV, {n_sm:.0f} events expected")
    fit = solve.fit_free_mass(data["counts"], x, data["edges"], data["widths"], sigma, params["m_start"], n_sm)
    m_pdg, err_pdg = tensors.M_H_PDG
    print(f"fit: m_H = {fit['m_h']:.2f} ± {fit['m_err']:.2f} GeV   (PDG 2024: {m_pdg} ± {err_pdg})")
    print(f"     signal strength μ = σ/σ_SM = {fit['mu']:.2f} ± {fit['mu_err']:.2f},  local Z = √q₀ = {fit['z']:.2f}σ")
    shape_sm = equation.signal_shape(data["edges"], torch.tensor(fit["m_h"], dtype=torch.float64), sigma)
    b_fit = data["widths"] * equation.background(fit["theta"], x)
    z_exp = equation.significance(solve.q0_known_background(b_fit + n_sm * shape_sm, b_fit, n_sm * shape_sm))
    print(f"     expected for the Standard Model (Asimov, μ = 1): Z = {z_exp.item():.2f}σ")

    # 2. local significance against mass, and background-only toys for the look-elsewhere effect
    masses = tensors.mass_grid(lo + 5, hi - 10, params["scan_step"], dev)  # (M,)
    q0_obs = solve.q0_scan(data["counts"][None], x, data["edges"], data["widths"], masses, sigma)[0]  # (M,)
    best = int(q0_obs.argmax())
    print(f"scan: largest local excess {math.sqrt(q0_obs[best].item()):.2f}σ at {masses[best].item():.1f} GeV")
    theta_b, _, _ = solve.fisher_fit(data["counts"][None], x, data["widths"])
    toys = tensors.toys(data["widths"] * equation.background(theta_b[0], x), params["n_toys"], seed=7)  # (T, N)
    q0_toys = solve.q0_scan(toys, x, data["edges"], data["widths"], masses, sigma)  # (T, M)
    k125 = int((masses - 125.0).abs().argmin())
    q_obs = q0_obs[k125].item()
    lee = solve.global_p_value(q0_toys, q_obs)
    zero = (q0_toys[:, k125] == 0).double().mean().item()
    above = (q0_toys[:, k125] > 1).double().mean().item()
    wilks = 0.5 * math.erfc(math.sqrt(0.5))
    print(f"toys at 125 GeV: q₀ = 0 in {zero:.0%} (Wilks: 50%), P(q₀ > 1) = {above:.3f} (Wilks: {wilks:.3f})")
    print(
        f"look-elsewhere: local p₀(125 GeV) = {lee['p_local']:.2e}; global p ≈ {lee['p_global_gv']:.2e} "
        f"(Gross–Vitells, {lee['n_up']:.2f} upcrossings of q₀ = 0.5); "
        f"toys beating it anywhere: {lee['p_global_toys']:.1%}"
    )

    # 3. the Mexican hat that gives the Higgs its mass
    mu2, lam = tensors.potential_parameters(fit["m_h"], tensors.V_EW)
    print(
        f"Mexican hat: v = (√2 G_F)^(−1/2) = {tensors.V_EW:.2f} GeV, λ = m_H²/2v² = {lam:.4f}, "
        f"minimum at |φ| = v/√2 = {tensors.V_EW / math.sqrt(2):.1f} GeV, "
        f"m_H = √(2λ) v = {math.sqrt(2 * lam) * tensors.V_EW:.2f} GeV"
    )

    return plot.animate(
        data, x, sigma, n_sm, fit, masses, q0_obs, q0_toys, k125, lee, (mu2, lam), out_dir / "higgs_boson.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/higgs_boson/equation.py
r"""Physics → tensor translation.

Diphoton events in a mass bin n are Poisson distributed around a smooth background plus a narrow Higgs peak:

    ν_n = w_n \exp(X_{nk} θ_k) + μ\, N_{SM}\, S_n(m_H, σ),          (k summed)
    S_n = Φ((m_{n+1} − m_H)/σ) − Φ((m_n − m_H)/σ)                   (Gaussian peak integrated over the bin)
    −\ln L = \sum_n (ν_n − k_n \ln ν_n)                              (Poisson likelihood, constants dropped)

The discovery test statistic is the profile likelihood ratio q₀ = −2 ln[L(μ = 0, \hat{\hat θ}) / L(\hat μ, \hat θ)]
for \hat μ ≥ 0 (else 0). By Wilks' theorem it follows ½δ(0) + ½χ²₁ without a signal, so Z = √q₀.

The Higgs field is a complex doublet; its neutral component φ sits in the Mexican-hat potential

    V(φ) = −μ²|φ|² + λ|φ|⁴,     minimum at |φ| = v/√2 with v = \sqrt{μ²/λ},     m_H² = 2λ v².
"""

import math

import torch


def background(theta: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""B_n = \exp(X_{nk} θ_k) per GeV  (k summed).   theta: (..., K), x: (N, K) → (..., N)"""
    return torch.exp(torch.einsum("nk,...k->...n", x, theta))


def signal_shape(edges: torch.Tensor, m_h: torch.Tensor, sigma: float) -> torch.Tensor:
    r"""S_n: fraction of a Gaussian peak at m_H falling in bin n.   edges: (N+1,), m_h: (...) → (..., N)"""
    z = (edges - m_h[..., None]) / (sigma * math.sqrt(2))
    cdf = 0.5 * torch.erf(z)
    return cdf[..., 1:] - cdf[..., :-1]


def expected(theta, n_s, x, widths, shape) -> torch.Tensor:
    r"""ν_n = w_n B_n + n_s S_n: expected events per bin.   → (..., N)"""
    return widths * background(theta, x) + n_s[..., None] * shape


def nll(nu: torch.Tensor, counts: torch.Tensor) -> torch.Tensor:
    r"""−ln L = Σ_n (ν_n − k_n ln ν_n) summed over the last axis.   → (...)"""
    return (nu - counts * torch.log(nu)).sum(-1)


def significance(q0: torch.Tensor) -> torch.Tensor:
    r"""Local significance Z = √q₀ (Wilks, one parameter of interest)."""
    return q0.clamp_min(0).sqrt()


def p_value(q0: torch.Tensor) -> torch.Tensor:
    r"""One-sided p₀ = 1 − Φ(√q₀) = ½ erfc(√(q₀/2))."""
    return 0.5 * torch.erfc((q0.clamp_min(0) / 2).sqrt())


def mexican_hat(phi: torch.Tensor, mu2: float, lam: float) -> torch.Tensor:
    r"""V(φ) = −μ²|φ|² + λ|φ|⁴ for a complex field φ (any shape)."""
    r2 = (phi * phi.conj()).real
    return -mu2 * r2 + lam * r2**2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/higgs_boson/tensors.py
"""Build the concrete tensors: the diphoton mass histogram, the background design matrix, toys and constants.

data/atlas_opendata_13tev_myy.csv is the m_γγ spectrum of 10 fb⁻¹ of real 13 TeV ATLAS collisions (2016) from the
ATLAS Open Data 2020 release (CERN Open Data record 15006, doi:10.7483/OPENDATA.ATLAS.B5BJ.3SGS, CC0), selected as in
the ATLAS H → γγ outreach analysis, in 0.5 GeV bins from 100 to 160 GeV. The column sm_signal is the Standard Model
H(125) → γγ expectation from the matching ATLAS simulation with the same cuts.
"""

import math
from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "atlas_opendata_13tev_myy.csv"
DTYPE = torch.float64
G_F = 1.1663788e-5  # Fermi constant, GeV⁻² (PDG 2024)
V_EW = (math.sqrt(2) * G_F) ** -0.5  # electroweak vacuum expectation value v ≈ 246.22 GeV
M_H_PDG = (125.20, 0.11)  # PDG 2024 world average Higgs mass, GeV
LUMI = 10.0  # fb⁻¹


def load_histogram(path: Path = DATA, m_min: float = 105.0, m_max: float = 160.0, device=None) -> dict:
    """Read the binned spectrum and keep the bins inside [m_min, m_max].

    Returns edges (N+1,), centers (N,), widths (N,), counts (N,) and sm_signal (N,) as float64 tensors. Any CSV with
    columns m_low, m_high, events (and optionally sm_signal) works; lines starting with # are comments.
    """
    lines = [ln for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip() and ln[0] != "#"]
    header = [h.strip() for h in lines[0].split(",")]
    rows = torch.tensor([[float(v) for v in ln.split(",")] for ln in lines[1:]], dtype=DTYPE)
    col = {name: rows[:, header.index(name)] for name in header}
    keep = (col["m_low"] >= m_min - 1e-9) & (col["m_high"] <= m_max + 1e-9)
    lo, hi = col["m_low"][keep], col["m_high"][keep]
    signal = col.get("sm_signal", torch.zeros_like(col["events"]))[keep]
    out = {
        "edges": torch.cat([lo, hi[-1:]]),
        "centers": (lo + hi) / 2,
        "widths": hi - lo,
        "counts": col["events"][keep],
        "sm_signal": signal,
    }
    return {k: v.to(device) for k, v in out.items()}


def design(centers: torch.Tensor, m_min: float, m_max: float, degree: int = 3) -> torch.Tensor:
    """X_{nk} = x_n^k with x = (m − m_min)/(m_max − m_min) ∈ [0, 1]: the background is exp(X θ). → (N, K)"""
    x = (centers - m_min) / (m_max - m_min)
    return x[:, None] ** torch.arange(degree + 1, dtype=centers.dtype, device=centers.device)


def mass_grid(m_lo: float, m_hi: float, step: float, device=None) -> torch.Tensor:
    """Hypothesised Higgs masses for the scan: (M,)."""
    return torch.arange(m_lo, m_hi + 1e-9, step, dtype=DTYPE, device=device)


def toys(expected: torch.Tensor, n: int, seed: int = 0) -> torch.Tensor:
    """n Poisson pseudo-experiments drawn from the expected counts (N,): (n, N), fixed seed."""
    gen = torch.Generator().manual_seed(seed)
    return torch.poisson(expected.cpu().expand(n, -1).contiguous(), generator=gen).to(expected.device)


def thinned(counts: torch.Tensor, fractions: torch.Tensor, seed: int = 0) -> torch.Tensor:
    """The histogram as it fills up: nested binomial subsamples holding a fraction f_j of the events. → (J, N)"""
    gen = torch.Generator().manual_seed(seed)
    counts = counts.cpu()
    out, have, prev = [], torch.zeros_like(counts), 0.0
    for f in fractions.tolist():
        p = (f - prev) / max(1.0 - prev, 1e-12)
        rest = counts - have
        have = have + torch.binomial(rest, torch.full_like(rest, min(max(p, 0.0), 1.0)), generator=gen)
        out.append(have.clone())
        prev = f
    return torch.stack(out)


def potential_parameters(m_h: float, v: float) -> tuple[float, float]:
    """μ² and λ of V = −μ²|φ|² + λ|φ|⁴ that give vacuum v and Higgs mass m_H:  μ² = m_H²/2, λ = m_H²/(2v²)."""
    return m_h**2 / 2, m_h**2 / (2 * v**2)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/higgs_boson/solve.py
r"""Fits: batched Fisher scoring for the mass scan and the toys, and autograd (Adam + L-BFGS) for the free-mass fit.

Fisher scoring (Newton's method with the expected Hessian) for p = (θ_k, n_s):
    J_{bnj} = ∂ν_{bn}/∂p_j = (w_n B_{bn} X_{nk},  S_{bn}),
    g_{bj} = J_{bnj} (1 − k_{bn}/ν_{bn}),     I_{bjl} = J_{bnj} J_{bnl} / ν_{bn}      (n summed),
    p ← p − I⁻¹ g,
run for every pseudo-experiment and every hypothesised mass at once.
"""

import torch

from problems._common.progress import steps as progress_steps

from . import equation


def _start(counts: torch.Tensor, x: torch.Tensor, widths: torch.Tensor) -> torch.Tensor:
    """Least-squares fit of ln(counts/width) by the polynomial: a good first guess for θ. → (B, K)"""
    target = torch.log((counts + 0.5) / widths)  # (B, N)
    return torch.linalg.lstsq(x, target.T).solution.T


def fisher_fit(counts, x, widths, shape=None, iterations: int = 12):
    """Maximum-likelihood θ (and signal yield n_s if a shape is given) for a batch of histograms.

    counts: (B, N), x: (N, K), widths: (N,), shape: (B, N) or None (background only).
    Returns θ (B, K), n_s (B,) and −ln L (B,).
    """
    theta = _start(counts, x, widths)
    n_s = torch.zeros(counts.shape[0], dtype=counts.dtype, device=counts.device)
    sig = torch.zeros_like(counts) if shape is None else shape.expand_as(counts)
    for _ in range(iterations):
        b = widths * equation.background(theta, x)  # (B, N)
        nu = b + n_s[:, None] * sig
        jac = torch.cat([b[..., None] * x, sig[..., None]], dim=-1)  # (B, N, K+1)
        if shape is None:
            jac = jac[..., :-1]
        g = torch.einsum("bnj,bn->bj", jac, 1 - counts / nu)
        info = torch.einsum("bnj,bnl->bjl", jac / nu[..., None], jac)  # I = Jᵀ diag(1/ν) J
        step = torch.linalg.solve(info, g)  # (B, K+1)
        theta = theta - step[:, : x.shape[1]]
        if shape is not None:
            n_s = n_s - step[:, -1]
    nu = widths * equation.background(theta, x) + n_s[:, None] * sig
    return theta, n_s, equation.nll(nu, counts)


def q0_scan(counts, x, edges, widths, masses, sigma: float):
    """Discovery statistic q₀ at every hypothesised mass for every histogram.  counts: (B, N) → (B, M)"""
    n_b, n_m = counts.shape[0], masses.shape[0]
    _, _, nll_b = fisher_fit(counts, x, widths)  # (B,) background only
    shapes = equation.signal_shape(edges, masses, sigma)  # (M, N)
    rep = counts[:, None, :].expand(n_b, n_m, -1).reshape(n_b * n_m, -1)  # (B·M, N)
    sig = shapes[None].expand(n_b, -1, -1).reshape(n_b * n_m, -1)
    _, n_s, nll_sb = fisher_fit(rep, x, widths, sig)
    q0 = 2 * (nll_b[:, None] - nll_sb.reshape(n_b, n_m))
    return torch.where(n_s.reshape(n_b, n_m) > 0, q0.clamp_min(0), torch.zeros_like(q0))


def q0_known_background(counts, b, s, iterations: int = 30) -> torch.Tensor:
    """q₀ when the background b (N,) is known: profile only the signal strength μ of ν = b + μ s (Newton in μ)."""
    mu = torch.zeros((), dtype=counts.dtype)
    for _ in range(iterations):
        nu = b + mu * s
        grad = (s * (1 - counts / nu)).sum()
        hess = (counts * s**2 / nu**2).sum()
        mu = mu - grad / hess
    q0 = 2 * (equation.nll(b, counts) - equation.nll(b + mu * s, counts))
    return q0 if mu > 0 else torch.zeros_like(q0)


def fit_free_mass(counts, x, edges, widths, sigma: float, m_start: float, n_sm: float, adam_steps: int = 150):
    """Fit θ, signal strength μ and m_H to one histogram with autograd; errors from the Hessian of −ln L.

    Adam (the path is kept for the animation) then L-BFGS polishes. n_sm is the Standard Model signal expectation,
    so μ = n_s / n_sm. Returns floats m_h, m_err, mu, mu_err, z (local, √q₀), theta (K,) and the path
    (S, K + 2) of (m_H, μ, θ).
    """
    counts = counts.reshape(1, -1)
    theta0, _, nll_b = fisher_fit(counts, x, widths)
    k = x.shape[1]
    # free parameters: θ (K), μ, (m_H − 125)/2  →  all of order one for Adam
    p = torch.cat([theta0[0], counts.new_tensor([0.5, (m_start - 125.0) / 2])]).requires_grad_(True)

    def loss(p):
        m_h = 125.0 + 2 * p[k + 1]
        shape = equation.signal_shape(edges, m_h, sigma)
        nu = equation.expected(p[:k], p[k] * n_sm, x, widths, shape)
        return equation.nll(nu, counts[0])

    opt = torch.optim.Adam([p], lr=0.03)
    path = []
    for _ in progress_steps(adam_steps, "Adam"):
        opt.zero_grad()
        value = loss(p)
        value.backward()
        opt.step()
        path.append(torch.cat([p.detach()[k + 1 :] * 2 + 125.0, p.detach()[: k + 1].roll(1)]))
    lbfgs = torch.optim.LBFGS([p], lr=1.0, max_iter=300, tolerance_grad=1e-10, line_search_fn="strong_wolfe")

    def closure():
        lbfgs.zero_grad()
        value = loss(p)
        value.backward()
        return value

    lbfgs.step(closure)
    best = p.detach().clone()
    path.append(torch.cat([best[k + 1 :] * 2 + 125.0, best[: k + 1].roll(1)]))
    cov = torch.linalg.inv(torch.autograd.functional.hessian(loss, best))  # −ln L: covariance = H⁻¹
    q0 = 2 * (nll_b[0] - loss(best)).detach().clamp_min(0)
    return {
        "m_h": 125.0 + 2 * best[k + 1].item(),
        "m_err": 2 * cov[k + 1, k + 1].sqrt().item(),
        "mu": best[k].item(),
        "mu_err": cov[k, k].sqrt().item(),
        "z": equation.significance(q0).item(),
        "theta": best[:k],
        "path": torch.stack(path),  # (S, K + 2) rows (m_H, μ, θ_0 … θ_{K−1})
    }


def fit_resolution(edges, sm_signal, m_start: float = 125.0) -> tuple[float, float]:
    """Mass and width of a Gaussian fitted (least squares, autograd) to the simulated signal histogram."""
    total = sm_signal.sum()
    p = sm_signal.new_tensor([m_start, 1.5]).requires_grad_(True)
    opt = torch.optim.LBFGS([p], lr=1.0, max_iter=200, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        model = total * equation.signal_shape(edges, p[0], p[1].abs())
        value = ((model - sm_signal) ** 2).sum()
        value.backward()
        return value

    opt.step(closure)
    return p[0].item(), p[1].abs().item()


def upcrossings(q0: torch.Tensor, level: float) -> torch.Tensor:
    """Number of times each scan q₀(m) crosses `level` upwards.  q0: (B, M) → (B,)"""
    above = q0 > level
    return (above[:, 1:] & ~above[:, :-1]).sum(-1)


def global_p_value(q0_toys: torch.Tensor, q0_obs: float, level: float = 0.5) -> dict:
    """Look-elsewhere effect from background-only toy scans.

    Direct estimate: the share of toys whose largest q₀ anywhere beats the observed one. Gross–Vitells (2010):
    p_global ≈ p_local + ⟨N_up(u₀)⟩ e^{−(q₀ − u₀)/2}, with the mean number of upcrossings counted in the toys.
    """
    q_max = q0_toys.max(-1).values
    n_up = upcrossings(q0_toys, level).double().mean()
    q = torch.tensor(q0_obs, dtype=torch.float64)
    p_local = equation.p_value(q)
    p_gv = p_local + n_up * torch.exp(-(q - level) / 2)
    return {
        "p_local": p_local.item(),
        "p_global_toys": (q_max >= q0_obs).double().mean().item(),
        "p_global_gv": p_gv.clamp(max=1).item(),
        "n_up": n_up.item(),
        "q_max": q_max,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/higgs_boson/plot.py
"""The histogram fills up, the fit walks to the peak, then the local significance is scanned against mass.

Left: m_γγ spectrum with the fit, data minus background, local p₀ against mass. Right: the Mexican-hat potential
and the background-only toys (q₀ at 125 GeV against ½χ²₁, and the largest q₀ anywhere: the look-elsewhere effect).
"""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import equation, tensors

BLUE, RED, GREY, AMBER, GREEN = "#2563eb", "#dc2626", "#9ca3af", "#f59e0b", "#16a34a"
N_FILL, N_FIT, N_SCAN = 22, 16, 22


def _model(row: torch.Tensor, x, edges, widths, sigma: float, n_sm: float):
    """Background and signal+background (per bin) for one path row (m_H, μ, θ...)."""
    b = widths * equation.background(row[2:], x)
    s = row[1] * n_sm * equation.signal_shape(edges, row[0], sigma)
    return b.numpy(), (b + s).numpy()


def animate(data, x, sigma, n_sm, fit, masses, q0_obs, q0_toys, k125, lee, potential, path: Path) -> Path:
    data = {k: v.cpu() for k, v in data.items()}
    x = x.cpu()
    c, counts, widths, edges = data["centers"].numpy(), data["counts"], data["widths"], data["edges"]
    fractions = torch.linspace(0.03, 1.0, N_FILL, dtype=torch.float64)
    filling = tensors.thinned(counts, fractions, seed=1).numpy()  # (J, N)
    steps = fit["path"]
    pick = np.unique(np.geomspace(1, len(steps), N_FIT).astype(int) - 1)
    models = [_model(steps[i], x, edges, widths, sigma, n_sm) for i in pick]
    b_final, sb_final = _model(steps[-1], x, edges, widths, sigma, n_sm)
    m, p0 = masses.cpu().numpy(), equation.p_value(q0_obs.cpu()).numpy()
    q_toys = q0_toys.cpu()

    fig = plt.figure(figsize=(10, 6.6))
    grid = fig.add_gridspec(3, 2, width_ratios=[2.1, 1], height_ratios=[2, 1, 1.3])
    ax_h = fig.add_subplot(grid[0, 0])
    ax_r = fig.add_subplot(grid[1, 0], sharex=ax_h)
    ax_p = fig.add_subplot(grid[2, 0], sharex=ax_h)
    ax_v = fig.add_subplot(grid[0, 1])
    ax_t = fig.add_subplot(grid[1:, 1])

    scale = counts.max().item() * 1.12
    ax_h.set_ylim(0, scale)
    ax_h.set_xlim(edges[0].item(), edges[-1].item())
    ax_h.set_ylabel(f"events / {widths[0].item():g} GeV")
    pts = ax_h.errorbar(c, filling[0], yerr=np.sqrt(filling[0]), fmt="o", ms=2, color="#111827", lw=0.8)
    (line_b,) = ax_h.plot([], [], color=BLUE, lw=1.5, ls="--", label="background")
    (line_sb,) = ax_h.plot([], [], color=RED, lw=2, label="signal + background")
    ax_h.legend(loc="upper right", fontsize=8)
    title = ax_h.set_title("", fontsize=10)
    info = ax_h.text(0.02, 0.05, "", transform=ax_h.transAxes, fontsize=8)

    resid = counts.numpy() - b_final
    ax_r.axhline(0, color=BLUE, lw=1, ls="--")
    ax_r.set_ylim(-4.5 * math.sqrt(b_final.max()), 4.5 * math.sqrt(b_final.max()))
    ax_r.set_ylabel("data − bkg")
    resid_pts = ax_r.errorbar(c, resid * 0, yerr=np.sqrt(counts.numpy()), fmt="o", ms=2, color="#111827", lw=0.6)
    (line_r,) = ax_r.plot([], [], color=RED, lw=2)
    resid_pts[0].set_visible(False)
    for bar in resid_pts[2]:
        bar.set_visible(False)

    ax_p.set_yscale("log")
    ax_p.set_ylim(min(p0.min(), 1e-3) / 5, 1.0)
    ax_p.set_ylabel("local $p_0$")
    ax_p.set_xlabel(r"diphoton mass $m_{\gamma\gamma}$ (GeV)")
    for z in (1, 2, 3):
        level = 0.5 * math.erfc(z / math.sqrt(2))
        ax_p.axhline(level, color=GREY, lw=0.8, ls=":")
        ax_p.text(edges[-1].item() - 0.5, level, f"{z}σ", fontsize=7, ha="right", va="bottom", color="#6b7280")
    ax_p.axvline(tensors.M_H_PDG[0], color=AMBER, lw=1)
    (line_p,) = ax_p.plot([], [], color=BLUE, lw=2)
    plt.setp(ax_h.get_xticklabels(), visible=False)
    plt.setp(ax_r.get_xticklabels(), visible=False)

    # Mexican hat: V along rays φ = r e^{iα} of the complex plane (all phases give the same curve)
    mu2, lam = potential
    r = torch.linspace(-265, 265, 401, dtype=torch.float64)
    alpha = torch.tensor([0.0, 1.0, 2.5], dtype=torch.float64)
    phi = r[None, :] * torch.exp(1j * alpha[:, None])  # (3, R) complex
    pot = equation.mexican_hat(phi, mu2, lam).numpy() / 1e8  # (3, R) in 10⁸ GeV⁴
    vac = tensors.V_EW / math.sqrt(2)
    v_min = -lam * tensors.V_EW**4 / 4 / 1e8
    for k, ls in enumerate(["-", "--", ":"]):
        ax_v.plot(r, pot[k], color=BLUE, lw=2 if k == 0 else 1.2, ls=ls)
    h = np.linspace(-60, 60, 50)  # φ = (v + h)/√2: V ≈ V_min + ½ m_H² h²
    m_h2 = 2 * lam * tensors.V_EW**2
    for sign in (-1, 1):
        ax_v.plot(sign * (vac + h / math.sqrt(2)), v_min + 0.5 * m_h2 * h**2 / 1e8, color=RED, lw=1.5)
    ax_v.plot([-vac, vac], [v_min, v_min], "o", color=RED, ms=4)
    ax_v.plot(0, 0, "o", color=AMBER, ms=4)
    ax_v.set_ylim(1.3 * v_min, 0.9)
    ax_v.set_xlabel(r"$r$ along $\phi = r\,e^{i\alpha}$ (GeV)", fontsize=8)
    ax_v.set_ylabel(r"$V$ ($10^8$ GeV$^4$)", fontsize=8)
    ax_v.tick_params(labelsize=7)
    ax_v.set_title(
        rf"$V=-\mu^2|\phi|^2+\lambda|\phi|^4$, $\lambda$ = {lam:.3f}"
        + "\n"
        + rf"minimum $|\phi| = v/\sqrt{{2}}$ = {vac:.0f} GeV; red: $\frac{{1}}{{2}}m_H^2h^2$",
        fontsize=8,
    )

    # toys: q₀ at 125 GeV and the largest q₀ anywhere
    bins = np.linspace(0, 14, 29)
    q125 = q_toys[:, k125].numpy()
    ax_t.hist(q125[q125 > 0], bins=bins, color=BLUE, alpha=0.5, label=r"toys: $q_0$ at 125 GeV")
    ax_t.hist(lee["q_max"].numpy(), bins=bins, color=AMBER, alpha=0.5, label=r"toys: max $q_0$ over mass")
    grid_q = np.linspace(0.05, 14, 200)
    chi2 = np.exp(-grid_q / 2) / np.sqrt(2 * np.pi * grid_q)
    width = bins[1] - bins[0]
    ax_t.plot(grid_q, 0.5 * chi2 * len(q125) * width, color="#111827", lw=1.2, label=r"$\frac{1}{2}\chi^2_1$ (Wilks)")
    obs_line = ax_t.axvline(0, color=RED, lw=2, label=r"observed $q_0$(125)", visible=False)
    ax_t.set_xlim(0, bins[-1])
    ax_t.set_yscale("log")
    ax_t.set_ylim(0.5, len(q125))
    ax_t.set_xlabel(r"$q_0 = -2\ln\lambda(0)$")
    ax_t.legend(fontsize=7, loc="upper right")
    lee_text = ax_t.text(0.04, 0.04, "", transform=ax_t.transAxes, fontsize=7.5)

    def update(frame: int):
        if frame < N_FILL:
            f = fractions[frame].item()
            y = filling[frame]
            _set_errorbar(pts, c, y, np.sqrt(y))
            title.set_text(f"ATLAS Open Data, 13 TeV: {10 * f:.1f} fb⁻¹ collected")
            return
        _set_errorbar(pts, c, counts.numpy(), np.sqrt(counts.numpy()))
        if frame < N_FILL + N_FIT:
            j = min(frame - N_FILL, len(models) - 1)
            b, sb = models[j]
            row = steps[pick[j]]
            line_b.set_data(c, b)
            line_sb.set_data(c, sb)
            title.set_text(f"fitting with autograd: m_H = {row[0].item():.1f} GeV, μ = {row[1].item():.2f}")
            return
        line_b.set_data(c, b_final)
        line_sb.set_data(c, sb_final)
        resid_pts[0].set_visible(True)
        for bar in resid_pts[2]:
            bar.set_visible(True)
        _set_errorbar(resid_pts, c, resid, np.sqrt(counts.numpy()))
        line_r.set_data(c, sb_final - b_final)
        title.set_text("H → γγ: fit to 10 fb⁻¹ of real ATLAS data")
        info.set_text(
            f"$m_H$ = {fit['m_h']:.1f} ± {fit['m_err']:.1f} GeV,  μ = {fit['mu']:.2f} ± {fit['mu_err']:.2f},  "
            f"local Z = {fit['z']:.1f}σ"
        )
        k = int(round((frame - N_FILL - N_FIT + 1) / N_SCAN * len(m)))
        line_p.set_data(m[:k], p0[:k])
        if frame == N_FILL + N_FIT + N_SCAN - 1:
            obs_line.set_xdata([q0_obs[k125].item()])
            obs_line.set_visible(True)
            lee_text.set_text(
                f"local p₀(125) = {lee['p_local']:.1e}\nglobal p ≈ {lee['p_global_gv']:.1e} (Gross–Vitells)\n"
                f"{lee['p_global_toys']:.0%} of toys beat it somewhere"
            )

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(N_FILL + N_FIT + N_SCAN, N_FILL + N_FIT + N_SCAN), path, fps=12)


def _set_errorbar(container, x, y, yerr) -> None:
    """Move the points and vertical error bars of an errorbar() container."""
    line, _, (bars,) = container
    line.set_data(x, y)
    bars.set_segments([np.array([[xi, yi - ei], [xi, yi + ei]]) for xi, yi, ei in zip(x, y, yerr)])


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'm_min': 105.0,
    'm_max': 160.0,
    'degree': 3,
    'm_start': 125.0,
    'scan_step': 0.5,
    'n_toys': 300
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.higgs_boson")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.higgs_boson.main")
problem_dir = Path("problems/nuclear/higgs_boson")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
